# Exercice 2.1 : un ticket P1 exige une approbation humaine (45 min)

> Prérequis : J2-01 et J2-02. Notebook **autonome** : exécutez-le de haut en bas.

**Contexte :** un ticket P1 réveille l'astreinte. NordTech exige qu'un responsable N2 l'approuve avant sa création. Les autres priorités sont créées directement.

| Étape | Consigne |
|---|---|
| 1. Routage | après `qualifier` : P1 → `validation_humaine`, sinon → `creer_ticket` |
| 2. Validation | `validation_humaine` suspend le graphe avec `interrupt()` et propose le ticket ; réponse attendue : `{"action": "approuver" \| "rejeter", "par": …, "motif"?: …}` |
| 3. Création | `creer_ticket` appelle `ticketing.creer(...)` avec le `thread_id` comme **clé d'idempotence** et enregistre qui a validé |
| 4. Refus | `ticket_refuse` n'écrit rien dans le ticketing |
| 5. Tests | les 3 demandes fournies, chacune dans son thread `ex21-<initiales>-<lettre>`, session Langfuse = thread |
| 6. Rejeu | simulez une panne juste après la création : **rejouez** `creer_ticket` pour « a » depuis le checkpoint qui le précède (J2-01 §6) ; il ne doit exister qu'**un** ticket |

**Critères de réussite :** aucun ticket P1 sans validation · le ticket P1 approuvé porte `valide_par` · le rejet n'écrit rien · le rejeu de `creer_ticket` ne crée pas de doublon · aucun effet de bord avant `interrupt()`.

## 0. Mise en route et éléments fournis

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

In [ ]:
from typing import Literal, TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

from helpdesk_ops.donnees import CATEGORIES
from helpdesk_ops.ticketing import Ticketing
from helpdesk_ops.tri import creer_classifieur, estimer_priorite, file_attente, normaliser_categorie

classifieur = creer_classifieur(llm)
ticketing = Ticketing()   # outil de ticketing simulé, vide au départ


class EtatTicket(TypedDict, total=False):
    ticket: str            # le texte de la demande
    categorie: str
    priorite: str          # P1 ou P3 (règle de J1)
    validation: dict       # la décision humaine : {"action", "par", "motif"?}
    ticket_id: str


def qualifier(state: EtatTicket) -> dict:
    """Fourni : catégorie par le LLM (J1), priorité par la règle explicable (défi J1-01)."""
    brute = classifieur.invoke({"ticket": state["ticket"], "categories": ", ".join(CATEGORIES)})
    return {"categorie": normaliser_categorie(brute), "priorite": estimer_priorite(state["ticket"])}

## 1. Vos nœuds

In [ ]:
def router_priorite(state: EtatTicket) -> Literal["validation_humaine", "creer_ticket"]:
    ...  # TODO


def validation_humaine(state: EtatTicket) -> Command[Literal["creer_ticket", "ticket_refuse"]]:
    ...  # TODO : interrupt() avec le ticket proposé, puis Command(update=..., goto=...)


def creer_ticket(state: EtatTicket, config: RunnableConfig) -> dict:
    ...  # TODO : ticketing.creer(cle_idempotence=<thread_id>, ...) ; renvoyer ticket_id


def ticket_refuse(state: EtatTicket) -> dict:
    ...  # TODO

## 2. Le graphe

In [ ]:
# TODO : StateGraph(EtatTicket), nœuds, arêtes, compile(checkpointer=InMemorySaver())
circuit = ...

from IPython.display import Markdown, display
display(Markdown(f"```mermaid\n{circuit.get_graph().draw_mermaid()}\n```"))

## 3. Les tests

> 💡 **Indice pour l'étape 6.** Reprendre une seconde fois une demande déjà approuvée ne prouve rien : le thread est terminé, aucun nœud ne repart (essayez : le ticketing ne bouge pas, avec ou sans clé d'idempotence). Pour tester vraiment l'idempotence, il faut **rejouer le nœud qui écrit**, comme après une panne survenue juste après l'appel au ticketing. C'est le voyage dans le temps de J2-01 §6 :
>
> ```python
> avant = [h for h in circuit.get_state_history(config_de("a")) if h.next == ("creer_ticket",)][-1]
> circuit.invoke(None, {**avant.config, "callbacks": [langfuse_handler], "run_name": "ex21-rejeu"})
> ```
>
> **Contre-épreuve (facultative) :** dans `creer_ticket`, remplacez un instant la clé par `str(uuid.uuid4())` (avec `import uuid`), puis réexécutez les sections 0 à 3 : le ticketing et le checkpointer repartent de zéro. Après le rejeu, « a » a **deux** tickets. Remettez ensuite le `thread_id`.

In [ ]:
DEMANDES = {
    "a": "URGENT : l'ERP de production est bloqué pour toute la compta",   # P1 → approuvé
    "b": "Production arrêtée : plus aucun scanner ne fonctionne à l'entrepôt",  # P1 → rejeté
    "c": "Je voudrais un deuxième écran pour mon poste",                     # P3 → créé directement
}

INITIALES = "xx"   # TODO


def config_de(lettre: str) -> dict:
    thread = f"ex21-{INITIALES}-{lettre}"
    return {"configurable": {"thread_id": thread}, "callbacks": [langfuse_handler], "run_name": "ex21"}


# TODO :
# - lancer les 3 demandes (version="v2") et afficher les interruptions éventuelles
# - approuver « a » (par="resp-n2"), rejeter « b » avec un motif
# - rejouer creer_ticket pour « a » depuis son checkpoint (indice ci-dessus)
# - afficher ticketing.lister()

## 4. Dans Langfuse

1. **Sessions → `ex21-<initiales>-a`** : combien de traces ? Pourquoi ?
2. Où voit-on **qui** a validé ? Est-ce suffisant pour un audit ?
3. La trace `ex21-rejeu` : le nœud `creer_ticket` a-t-il bien été réexécuté ? Quel `ticket_id` a-t-il renvoyé ?